In [ ]:
# 약속 1. Data 규칙 부여 시 min, max etc. 는 _min = / _max 와 같이 사용할 것
# ex) data_***_min = data_rate.min()

#약속 2. 약어를 쓸 경우 각 단어의 앞 글자로만 구성할 것.
# Released_Year = ry

#약속 3. Extract = ext, Separate = spr, Filter = flt etc. 단어가 길 경우 앞의 모음 세 글자를 사용하는 것으로 약속 만약 겹칠 경우 끝자리를 달리하는 등의 규칙을 여기에 꼭 표기할 것.
#  Extend  = exd (추가), director = drc (추가)

#약속 4. 수정사항은 # 표기와 함께 간략히 적을 것 (ex. 명칭 변경, 숫자 변경 등)

# Step 1
#import matplotlib.pyplot as plt
import numpy as np
import csv
with open("IMDB top 1000.csv", 'r', encoding='utf-8') as f: #file_name 불필요 (수정_wk)
    reader = csv.reader(f)
    next(reader)
    data_list = [row for row in reader]
data_array = np.array(data_list)
#2
data_title = data_array[:,1:2]
data_genre = data_array[:,4]
data_rate = data_array[:,5:6]
released_year = np.array([year[0][-5:-1] for year in data_title])
data_ry = released_year.reshape(-1,1) #array로 변경
#2-1 
data_cast = data_array[:,8:9]
cast_spr = np.char.split(data_cast[:,0],' | ') # spr : Separate
# Cast_Director 분리
data_drc = np.array([drc_row[0].replace('Director: ','').replace('Directors: ','').strip() for drc_row in cast_spr])
data_drc = data_drc.reshape(-1,1)
#print(data_drc)
#2-2
#Cast_Star 분리
data_star = np.array([star_row[1].replace('Stars: ','').strip() for star_row in cast_spr])
data_star = data_star.reshape(-1,1)
#print(data_star)
#3 
data_stack = np.column_stack((data_title, data_genre, data_rate,data_ry,data_drc,data_star))
column_header_step1 = "Title,Genre,Rate,Released_Year,Director,Star"
data_stack_array = np.array(data_stack)

#4 
fil_NaN = np.all(data_stack_array != '', axis=1)
data_filter = data_stack_array[fil_NaN]
data_step1 = data_filter
#print(data_step1)
# Step 2 -------------------------------------------------------------------------------------
# 전체 영화 개수, 평균 평점, 최고 평점, 최저 평점 출력
print("\n=== Step 2: 기본적인 데이터 탐색 ===")

#rt_idx : Rate index(평점이 있는 열의 위치 번호라는 의미)
#astype은 float로 바꾸라는 코드. 기존 str이라 float로 바꾸어야 계산이 가능.
rate_idx = 2 #Rate index
data_rate = data_step1[:, rate_idx].astype(float)

# 전체 영화 개수
data_mov_count = data_step1.shape[0]

# 평균, 최고, 최저 평점 계산
# avg는 값이 8.0975로 나와서 Round 사용하여 8.1로 계산되게끔 함.
data_rate_avg = round(data_rate.mean(), 1)
data_rate_max = data_rate.max()
data_rate_min = data_rate.min()

#출력
print(f"총 영화 개수: {data_mov_count}")
print(f"평점 평균: {data_rate_avg}")
print(f"최고 평점: {data_rate_max}")
print(f"최저 평점: {data_rate_min}")
print("-"*60)

# Step 3 -------------------------------------------------------------------------------------
print("\n=== Step 3: 평점이 높은 영화 찾기 ===")
# : 평점이 높은 영화 찾기

data_rate_max = data_rate.max()

# np.where를 통해 "조건을 만족하는 데이터의 위치" 찾기 가능
top_idx = np.where(data_rate == data_rate_max)[0]

print("최고 평점 영화 목록:")

for i in top_idx:
    title = data_step1[i, 0]
    rate = data_step1[i, 2]

#출력
print(f"{title} - 평점: {rate}")
print("-"*60)
# Step 4 -------------------------------------------------------------------------------------
print("\n=== Step 4: 특정 장르별 평균 평점 분석 ===")
#4-1 필요한 열 추출 및 타입 지정
#data_rte = data_step1[:,2].astype(float) # 위의 data_rate 사용

#4-2 장르 분리 및 리스트에 모으기
genre_all_lst = []
#for gnr_row in data_gnr:
for genre_row in data_genre:
    genre_spr = [genre_itm.strip() for genre_itm in genre_row.split(",")]
    genre_all_lst.extend(genre_spr)

#4-3 장르 중복 제거
data_genre_all = np.array(genre_all_lst, dtype=object)
data_genre_uni = np.unique(data_genre_all)

#4-4 장르별 평균 평점 계산
genre_avg_lst = []
for genre_itm in data_genre_uni:
    data_genre_flt = (np.char.find(data_genre,genre_itm) >= 0)
    data_rate_avg = data_rate[data_genre_flt].mean()
    genre_avg_lst.append((genre_itm, data_rate_avg))
data_genre_avg = np.array(genre_avg_lst, dtype=object)

#4-5 평균 평점 기준으로 정렬
data_rate_avg_val = data_genre_avg[:,1].astype(float)
data_rate_avg_ord = np.argsort(data_rate_avg_val)[::-1]
data_genre_avg_srt = data_genre_avg[data_rate_avg_ord]

#출력
#for genre_itm, rate_avg in data_genre_avg_srt:
    #print(f"{genre_itm}: {float(rate_avg):.2f}")
print("-"*60)
# Step 5 -------------------------------------------------------------------------------------
print("\n=== Step 5: 연도별 평점 변화 분석 ===")
ry_idx = 3     #Releasd_year index (ry)

#연도와 평점을 숫자 타입으로 변환 과정
years = data_step1[:, ry_idx].astype(int)
rates = data_step1[:, rate_idx].astype(float)

#연도 고유값 추출(중복 년도 재거, 오름차순 표시)
unique_years = np.unique(years)

year_avg = []

for y in unique_years:
    #해당 연도의 평점만 표시
    flt = years == y
    avg_rate = rates[flt].mean()

    year_avg.append([y, avg_rate])

#출력
#for y, avg in year_avg:
    #print(f"{int(y)}년 average rating: {round(avg, 2)}")
print("-"*60)
# Step 6 -------------------------------------------------------------------------------------
print("\n=== Step 6: 시각화 ===")
#years_num = np.array(list(sorted_year.keys())).astype(int)
#years_avg = np.array(list(sorted_year.values()))

#plt.figure(figsize=(10, 5))
#plt.plot(years_num, years_avg, marker='o')
#plt.title('연도별 평균 평점 변화')
#plt.xlabel('연도')
#plt.ylabel('평균 평점')
#plt.grid(True)
#plt.xticks(years_num[::5], rotation=45)  # 5년 간격
#plt.tight_layout()
#plt.show()

# Step 7 -------------------------------------------------------------------------------------
# 7-1 프로젝트 확장 -1
# 함께 일한 배우와 감독의 조합에서 평균 평점 분석 [어떤 배우와 감독의 조합이 인기가 있었는지]

# 7-2 프로젝트 확장 -2
# 시대별(10년 단위)로 출시한 장르의 개수 분석 및 비교 [어떤 장르가 시대별로 인기가 있었는지]



In [ ]:
# Step 1
#import matplotlib.pyplot as plt
import numpy as np
import csv
with open("IMDB top 1000.csv", 'r', encoding='utf-8') as f: #file_name 불필요 (수정_wk)
    reader = csv.reader(f)
    next(reader)
    data_list = [row for row in reader]
data_array = np.array(data_list)
#2
data_title = data_array[:,1:2]
data_genre = data_array[:,4]
data_rate = data_array[:,5:6]
released_year = np.array([year[0][-5:-1] for year in data_title])
data_ry = released_year.reshape(-1,1) #array로 변경
#2-1 
data_cast = data_array[:,8:9]
cast_spr = np.char.split(data_cast[:,0],' | ') # spr : Separate
# Cast_Director 분리
data_drc = np.array([drc_row[0].replace('Director: ','').replace('Directors: ','').strip() for drc_row in cast_spr])
data_drc = data_drc.reshape(-1,1)
#print(data_drc)

#Cast_Star 분리
data_star = np.array([star_row[1].replace('Stars: ','').strip() for star_row in cast_spr])
data_star = data_star.reshape(-1,1)
#print(data_star)
#3 
data_stack = np.column_stack((data_title, data_genre, data_rate,data_ry,data_drc,data_star))
column_header_step1 = "Title,Genre,Rate,Released_Year,Director,Star"
data_stack_array = np.array(data_stack)

#4 
fil_NaN = np.all(data_stack_array != '', axis=1)
data_filter = data_stack_array[fil_NaN]
data_step1 = data_filter
#print(data_step1)